In [ ]:
%reload_ext autoreload
%autoreload 2

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

import os
import logging
import yaml
import h5py
import ipynbname

import pandas as pd

import ferdinand.utils as utl   
import ferdinand.sqlite_utils as sqlutl
import ferdinand.model_setup as modsetup
import ferdinand.preprocess_and_qc as ppqc

from pathlib import Path
from types import SimpleNamespace

from keras.optimizers import Adam

# Purpose and Documentation

This notebook provides a standardized structure for project notebooks, including path setup, logging, and metadata tracking. </br>
Use it as a starting point for new notebooks to ensure consistency, reproducibility, and clarity across the project.

# Definition / Settings

This section initializes the notebook environment: 
- loads the runtime configuration from _../config/config.yml_
- creates the required directory structure
- sets up logging and the SQLite connection

**Do not proceed past this section** </br>
Missing paths and an uninitialized database connection will cause subsequent cells to crash.

## General settings

In [ ]:
# load the configuration file
config_path = Path('../config/config.yml')
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

print(pd.DataFrame(config['settings'].items(), columns=['configuration parameter', 'set to']).to_markdown(tablefmt="pretty", index=False))

## Set up runtime paths

In [ ]:
# create directories as per configuration
dirs = SimpleNamespace(**config['directories'])
for key, value in vars(dirs).items():
    dirs.__dict__[key] = os.path.join(config['project_root'], config['data_dir'], config['run_name'], value)
    os.makedirs(dirs.__dict__[key], exist_ok=True)
    if os.path.exists(dirs.__dict__[key]):
        print(f" Directory created: {dirs.__dict__[key]}")
    else:
        print(f" ERROR: Directory does not exist: {dirs.__dict__[key]}")

## Store runtime configuration when necessary

In [ ]:
# copy the current '../config/config.yml' to '../reports/'
dest_filename = f"config_{config['run_name']}.yml"
dest_path = os.path.join(config['project_root'], config['reports_dir'], config['run_name'], dest_filename)
if not os.path.exists(dest_path): 
    commmand_string = f"cp {config_path} {dest_path}"
    os.system(commmand_string)

    if os.path.exists(dest_path): 
        print(f" Current runtime configuration \'{config_path}' saved to \'{dest_path}\'")
    else:
        print(f" ERROR: could not save configuration file to \'{dest_path}\'")

## Set up logger

In [ ]:
# setup logging
notebook_name = ipynbname.name()

log_path = os.path.join(config['project_root'], config['reports_dir'], config['run_name'], 'logs')
os.makedirs(log_path, exist_ok=True)

log_file = os.path.join(log_path, f"{notebook_name}.log")
print(f"Log file: {log_file}")

try:
    logger = utl.setup_logger(log_file, log_level=logging.INFO)
    print("Logger successfully set up.")
except Exception as e:
    print(f"Error setting up logger: {e}")

## Set up SQLite connection

In [ ]:
# expected SQLlite database file
sqlite_file = os.path.join(dirs.all_data_dir, config['sqlite_db'])

# if already exists, connect to it
if os.path.exists(sqlite_file):
    conn = sqlutl.connect_db(sqlite_file)
    if conn is not None:
        print(f"connected to database {sqlite_file}\n")
else: 
    print(f" ERROR: No SQLite database found!")

# Predict orientation angle

In [ ]:
# check if the expected AI model file exists and can be read as HDF5
try: 
    model_file = os.path.join(dirs.models_dir, config['angle_detection_model'])
    if not os.path.exists(model_file): 
        print(f"ERROR: no angle detection model {model_file} found!")
    else:
        print(f" model size: {os.path.getsize(model_file)}")
        try:
            with h5py.File(model_file, "r") as f: 
                print("keys in HDF5 file:", list(f.keys()))
        except Exception as e:
            print("ERROR: Not HDF5:\n", e)

except NameError as e:
    print(f"model path could not be set up: {e}")

In [ ]:
# load the model architecture and weights
model = modsetup.create_model()
model.compile(optimizer=Adam(learning_rate=0.0001), 
              loss='categorical_crossentropy', 
              metrics=[modsetup.angle_error])
if os.path.exists(model_file):
    model.load_weights(model_file)

## First angle prediction

In [ ]:
curated_images = utl.select_jpegs_by_columns(conn=conn, 
                                             db_table='impc_data', 
                                             filters={'preprocessed': 'yes'}, #{'keep': 'yes', 'overexposed': 'no', 'overall_overexposed': 'no'}, 
                                             source_path=dirs.preprocessed_dir) 
for _img in curated_images:
    angle = ppqc.predict_orientation_angle(conn=conn, 
                                           db_table='impc_data', 
                                           model=model, 
                                           image_file_path=_img,
                                           db_column_name='fst_angle_prediction')
    print(f" fst angle prediction for image {_img}: {angle}")

# Rotate images

In [ ]:
imgs_to_rotate = utl.select_jpegs_by_columns(conn=conn, 
                                             db_table='impc_data',
                                             filters={#'keep': 'yes', 
                                                      #'overexposed': 'no',
                                                      #'overall_overexposed': 'no', 
                                                      'rotated': 'no'}, 
                                             source_path=dirs.preprocessed_dir)
if len(imgs_to_rotate) == 0:
    imgs_to_rotate = utl.select_jpegs_by_columns(conn=conn, 
                                                 db_table='impc_data',
                                                 filters={'preprocessed': 'yes'}, #{
                                                        #'keep': 'yes', 
                                                        #'overexposed': 'no',
                                                        #'overall_overexposed': 'no'}, 
                                               source_path=dirs.preprocessed_dir)
if len(imgs_to_rotate) > 0:
    print(f" number of curated images with no exposure to rotate: {len(imgs_to_rotate)}")

    ppqc.load_and_rotate_images(conn, 
                                db_table='impc_data',
                                image_files=imgs_to_rotate, 
                                source_path=dirs.preprocessed_dir,
                                target_path=dirs.rotated_dir,
                                logger=logger)

# Create thumbnail gallery

In [ ]:
# select all rotated images
rows = sqlutl.select_rows_by_column(conn, 'impc_data', 'rotated', 'yes')

In [ ]:
# create thumbnails for all rotated images
utl.create_and_save_thumbnails(
    data=rows,
    centers=config['centers'],
    path2images=dirs.rotated_dir,
    path2thumbnails=dirs.rotated_thumbnails_dir
)

In [ ]:
# create HTML gallery for all rotated images
utl.create_thumbnail_gallery(
    data=rows,
    html_file_name='_rotated_images.html',
    title='Overview of Rotated Images',
    html_dir=os.path.join(config['project_root'], config['reports_dir'], config['run_name']),
    thumbnails_dir=dirs.rotated_thumbnails_dir,
    source_dir=dirs.rotated_dir)